Problemas identificados no dataset:

1. Colunas Desnecessárias e Valores Ausentes
registro_id: A coluna deve ser eliminada do dataset.
Dados Faltantes: Existem valores nulos nas colunas obrigatórias placa (3 nulos) e cpf (1 nulo), além de lacunas em marca_id, modelo_id e ano_modelo.

2. Inconsistência de Formatos (Datas e Strings)
ano_modelo: Presença de espaços em branco (ex: ' 2011-3  ') que impedem a padronização automática.

data_venda: O campo apresenta pelo menos quatro padrões diferentes (aaaa-mm-dd, dd/mm/aaaa, dd-mm-aaaa, aaaa/mm/dd), o que exige a conversão para o padrão único dd/mm/aaaa.

placa: Alguns registros contêm caracteres especiais como hifens (ex: DEF-4K56), violando a regra de conter apenas letras e números.

cliente: Nomes registrados com variações de caixa (MAIÚSCULAS, minúsculas e Capitalizada), o que dificulta a agregação por cliente.

3. Problemas com Campos Numéricos e Monetários
valor_venda, taxa_servico e valor_total:

Uso inconsistente de símbolos monetários ("R$").

Conflito no uso de separadores: em alguns registros o ponto é usado como separador de milhar (88.629,00) e em outros como separador decimal (88629.00).


km_rodados: Presença de textos (ex: "120000km") e separadores de milhar (120.000 ou 98-000), enquanto o requisito exige apenas números.

cpf: Presença de pontos e hifens (ex: 123.456.789-09). Deve ser limpo para conter apenas os dígitos numéricos.

4. Erros de Integridade e Domínio
Cálculo de valor_total: Existem divergências matemáticas. No registro 9, por exemplo, a soma de valor_venda (73.500,00) e taxa_servico (1.000,00) resulta em 74.500,00, mas o valor total gravado é 74.400,00.

estado: Existem siglas de estados inválidas como "XX" e "ZZ".

marca_id e modelo_id: Alguns campos contêm valores não numéricos (como "XX") que devem ser tratados para garantir a integridade referencial.


In [1]:
import pandas as pd
import requests
import re

In [2]:
df = pd.read_csv("carros-fipe.csv", sep=";")

In [26]:
df.loc[0, "cliente"] = "Maria Luiza"

In [4]:
df = df.drop(columns=["registro_id"])

In [5]:
descartados = pd.DataFrame()
descartados_endpoint = pd.DataFrame()

In [6]:
def limpar_valor(v):
    if pd.isna(v):
        return None
    v = str(v)
    v = re.sub(r"[^\d,]", "", v)
    v = v.replace(",", ".")
    return float(v)

df["valor_venda"] = df["valor_venda"].apply(limpar_valor)
df["taxa_servico"] = df["taxa_servico"].apply(limpar_valor)

In [7]:
df["valor_total"] = df["valor_venda"] + df["taxa_servico"]

In [8]:
df["km_rodados"] = df["km_rodados"].astype(str)
df["km_rodados"] = df["km_rodados"].str.replace(r"\D", "", regex=True)
df["km_rodados"] = pd.to_numeric(df["km_rodados"], errors="coerce")

In [10]:
df["cpf"] = df["cpf"].astype(str)
df["cpf"] = df["cpf"].str.replace(r"\D", "", regex=True)

In [9]:
df["placa"] = df["placa"].astype(str)
df["placa"] = df["placa"].str.replace(r"[^A-Za-z0-9]", "", regex=True)

In [11]:
df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce", dayfirst=True)

In [12]:
invalidos = df[(df["placa"] == "") | (df["cpf"] == "")]

descartados = pd.concat([descartados, invalidos])

df = df.drop(invalidos.index)

In [13]:
def consultar_fipe(row):

    marca = row["marca_id"]
    modelo = row["modelo_id"]
    ano = str(row["ano_modelo"]).strip()

    url = f"https://parallelum.com.br/fipe/api/v1/carros/marcas/{marca}/modelos/{modelo}/anos/{ano}"

    try:
        r = requests.get(url)

        if r.status_code != 200:
            return None, None

        dados = r.json()

        valor = dados["Valor"]
        valor = re.sub(r"[^\d,]", "", valor)

        marca_nome = dados["Marca"]

        return valor, marca_nome

    except:
        return None, None

In [14]:
df[["preco_fipe", "marca"]] = df.apply(
    lambda row: pd.Series(consultar_fipe(row)),
    axis=1
)

In [15]:
falha_endpoint = df[df["preco_fipe"].isna()]

descartados_endpoint = pd.concat([descartados_endpoint, falha_endpoint])

df = df.drop(falha_endpoint.index)

In [16]:
df["preco_fipe_num"] = df["preco_fipe"].str.replace(",", ".").astype(float)

In [17]:
df["lucro"] = df["preco_fipe_num"] - df["valor_venda"]

In [18]:
df["ano"] = df["ano_modelo"].str.split("-").str[0]

In [19]:
pivot = pd.pivot_table(
    df,
    values="valor_total",
    index="marca",
    columns="ano",
    aggfunc="sum"
)

In [20]:
df.to_csv("carros-fipe-saida.csv", index=False)

In [21]:
descartados.to_csv("carros-fipe-descarte.csv", index=False)

In [22]:
descartados_endpoint.to_csv("carros-fipe-descarte-endpoint.csv", index=False)

In [29]:
pivot.to_csv("total_por_marca.csv")
pivot = pivot.fillna(0)
pivot


ano,2011,2003,2010,2011,2020,2021
marca,,,,,,
Fiat,0.0,0.0,187560.0,0.0,0.000,0.0
GM - Chevrolet,0.0,74500.0,0.0,0.0,0.000,0.0
VW - VolksWagen,89829.0,0.0,0.0,89829.0,1288.629,17965800.0


In [24]:
df.head()
df.info()
df.columns

<class 'pandas.core.frame.DataFrame'>
Index: 10 entries, 0 to 42
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   marca_id        10 non-null     object        
 1   modelo_id       10 non-null     object        
 2   ano_modelo      10 non-null     object        
 3   data_venda      3 non-null      datetime64[ns]
 4   valor_venda     10 non-null     float64       
 5   taxa_servico    10 non-null     float64       
 6   valor_total     10 non-null     float64       
 7   km_rodados      10 non-null     int64         
 8   placa           10 non-null     object        
 9   cliente         10 non-null     object        
 10  cpf             10 non-null     object        
 11  estado          10 non-null     object        
 12  preco_fipe      10 non-null     object        
 13  marca           10 non-null     object        
 14  preco_fipe_num  10 non-null     float64       
 15  lucro        

Index(['marca_id', 'modelo_id', 'ano_modelo', 'data_venda', 'valor_venda',
       'taxa_servico', 'valor_total', 'km_rodados', 'placa', 'cliente', 'cpf',
       'estado', 'preco_fipe', 'marca', 'preco_fipe_num', 'lucro', 'ano'],
      dtype='object')